# 05 - Evaluator-Optimizer Workflow and Memory Testing

Test the Evaluator-Optimizer workflow, and implementation of self-reflection and learning capabilities of agent. 

Evaluator-Optimizer workflow will allow the system to:\
**Generate analysis → evaluate quality → refine using feedback** 

This notebook completes the following:
 1. Creates a memory storage located in `agent_memory.json` to allow the agent to store and learn from previous runs.
 2. Tests an Evaluator that allows the agent to self-reflect through strict requirements and guidelines.
 3. Tests an Optimizer that allows the agent to learn from previous runs and imrpove outputs. 
 4. Tests the full Evaluator-Optimizer workflow by generating analysis, evaluating quality, and refining using feedback through 3 iterations. 

## 1. Setup

Load data from notebooks 00-02 in order to gain access to pulled data from APIs and Databases. This does **not** complete the ingestion, preprocessing, new chain, or other agent functions aside from those explicitly listed above. 

This section also ensure that LLM keys are set up to use OpenAI model gpt-4o-mini to aid in the reflection and optimization qualities of the agent. 

Ensure that 
```text
LLM_API_KEY=your_key_here
LLM_MODEL=gpt-40-mini

# Optional for another OpenAI-compatible provider.
LLM_BASE_URL=
```


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

# Verify LLM configuration
print("LLM Model:", os.getenv("LLM_MODEL", "Not set"))
print("LLM Base URL:", os.getenv("LLM_BASE_URL", "Not set"))

# Import src modules
from src.memory import ResearchMemoryStore
from src.evaluator import evaluate_draft, EvaluationResult
from src.optimizer import optimize_draft
from src.eval_opt_workflow import run_evaluator_optimizer_workflow

print("Environment configured and all src modules imported successfully!")

LLM Model: gpt-4o-mini
LLM Base URL: 
Environment configured and all src modules imported successfully!


In [3]:
# Load previously made paths
daily_evidence_path = "../data/processed/daily_evidence.csv"
news_results_path = "../data/processed/news_research_results.csv"

# Load DataFrames
daily_df = pd.read_csv(daily_evidence_path)
news_df = pd.read_csv(news_results_path)

print(f"Loaded daily_evidence.csv: {daily_df.shape} rows")
print(f"Loaded news_research_results.csv: {news_df.shape} rows")

# Filter evidence for target symbol for testing purposes
TICKER = "AAPL"
aapl_daily = daily_df[daily_df["ticker"] == TICKER]
aapl_news = news_df[news_df["ticker"] == TICKER]

# Format raw evidence text for evaluation context
combined_evidence_text = f"""
=== DAILY MARKET & SEC EVIDENCE ===
{aapl_daily.to_string(index=False)}

=== PROCESSED NEWS RESEARCH RESULTS ===
{aapl_news.to_string(index=False)}
"""

print(f"\nCombined Evidence length for {TICKER}: {len(combined_evidence_text)} characters")

Loaded daily_evidence.csv: (58, 15) rows
Loaded news_research_results.csv: (3, 14) rows

Combined Evidence length for AAPL: 26448 characters


## 2. Test Memory

Ensure that agent stores memory at `agent_memory.json` by creating and saving a mock run to be retrieved. Display historical run data.

In [4]:
# Test path for memory store
TEST_MEMORY_PATH = "../data/processed/test_memory_store.json"

# Clean up previous test file if present
if os.path.exists(TEST_MEMORY_PATH):
    os.remove(TEST_MEMORY_PATH)

# Initialize JSON memory store
memory_store = ResearchMemoryStore(filepath=TEST_MEMORY_PATH)

# Test saving a mock run for AAPL
memory_store.save_run(
    ticker="AAPL",
    timestamp="2026-09-30 10:00:00",
    score=7,
    summary="Initial run lacked SEC Form 4 insider transaction details.",
    lessons_learned="Always emphasize Form 4 insider buying/selling trends."
)

# Retrieve and inspect memory context
aapl_memory = memory_store.get_ticker_memory("AAPL")
print("--- RETRIEVED MEMORY FOR AAPL ---")
print(aapl_memory)

assert "Form 4" in aapl_memory, "Memory retrieval check failed"

[MemoryStore] Saved run for AAPL to ../data/processed/test_memory_store.json.
--- RETRIEVED MEMORY FOR AAPL ---
=== HISTORICAL MEMORY FOR AAPL ===
Run #1 [2026-09-30 10:00:00] - Final Quality Score: 7/10
  Summary: Initial run lacked SEC Form 4 insider transaction details.
  Lessons Learned: Always emphasize Form 4 insider buying/selling trends.


## 3. Test Evaluator
Intentionally create a weak draft to ensure the evaluator catches mistakes, grades well, and makes clear note of failure points.

In [6]:
# Create an intentionally incomplete candidate draft to test evaluation
weak_draft = """
AAPL stock traded slightly higher over the past week. 
The news sentiment is generally positive. Revenue numbers look strong.
"""

print("Evaluating weak draft against evidence...")
eval_result: EvaluationResult = evaluate_draft(
    current_draft=weak_draft,
    raw_evidence=combined_evidence_text
)

print("\n--- EVALUATION RESULT ---")
print(f"Overall Score: {eval_result.overall_score}/10")
print(f"Passed Gate: {eval_result.passed}")

if not eval_result.passed:
    print(f"Reason: {eval_result.failure_reason}")

print(f"Factual Grounding: {eval_result.rubric.factual_grounding}/10")
print(f"Completeness: {eval_result.rubric.completeness}/10")
print(f"Missing Elements: {eval_result.missing_elements}")
print(f"Refinement Feedback:\n{eval_result.refinement_feedback}")

assert isinstance(eval_result, EvaluationResult), "Evaluator failed to return EvaluationResult model"

Evaluating weak draft against evidence...

--- EVALUATION RESULT ---
Overall Score: 5/10
Passed Gate: False
Reason: Factual grounding is low due to lack of specific stock price data and unsupported claims about revenue numbers. Completeness is also lacking as key metrics and insider transactions are not adequately addressed.
Factual Grounding: 4/10
Completeness: 5/10
Missing Elements: ['Specific stock prices for the past week', 'Detailed revenue numbers', 'Insider transaction details']
Refinement Feedback:
The report needs to include specific stock price data from the raw evidence, such as the closing prices for the relevant dates. Additionally, it should provide more context regarding the insider transactions and any relevant news articles that could impact the stock's performance. Clarifying the source of the 'strong revenue numbers' claim is essential, as it currently lacks support from the evidence provided.


## 4. Test Optimizer
Tests optimizer using evaluator feedback to ensure optimizer makes changes to previous draft and improves output based on feedback. 

In [7]:
print("Testing draft optimization using evaluator feedback...")

print("\n--- ORIGINAL DRAFT ---")
print(weak_draft)

refined_draft = optimize_draft(
    current_draft=weak_draft,
    failure_reason=getattr(eval_result, "failure_reason", "N/A"),
    feedback=eval_result.refinement_feedback,
    missing_elements=eval_result.missing_elements,
    raw_evidence=combined_evidence_text,
    overall_score=eval_result.overall_score,
    memory_context=aapl_memory
)

print("\n--- REFINED DRAFT OUTPUT ---")
print(refined_draft)

Testing draft optimization using evaluator feedback...

--- ORIGINAL DRAFT ---

AAPL stock traded slightly higher over the past week. 
The news sentiment is generally positive. Revenue numbers look strong.


--- REFINED DRAFT OUTPUT ---
## AAPL Stock Research Report

AAPL stock has shown fluctuations in trading over the past week. Below are the specific closing prices for the relevant dates:

- September 25, 2026: Closed at $341.070007
- September 28, 2026: Closed at $338.399994
- September 29, 2026: Closed at $329.399994
- September 30, 2026: Closed at $333.019989
- October 1, 2026: Closed at $329.609985

The news sentiment is mixed, with a recent downgrade from analysts. Notably, on September 25, 2026, an article titled "Apple: Why I'm Betting Against This Valuable Company (Rating Downgrade)" highlighted concerns regarding Apple's valuation, stating that Apple trades at a lofty 38x forward P/E ratio. This downgrade may indicate potential challenges for the company's stock performance

## 5. Test Evaluator-Optimizer Workflow
Complete two sets of 3 runs to test Evaluator-Optimizer workflow using AAPL data.

In [8]:
print("=== RUN 1: TESTING END-TO-END WORKFLOW LOOP ===")

MEMORY_FILE = "../data/processed/memory_store.json"

final_draft_run1, final_eval_run1, history_run1 = run_evaluator_optimizer_workflow(
    ticker=TICKER,
    initial_draft=weak_draft,
    raw_evidence=combined_evidence_text,
    max_iterations=3,
    memory_store_path=MEMORY_FILE
)

print("\n=== RUN 1 FINAL SUMMARY ===")
print(f"Total Iterations: {len(history_run1)}")
print(f"Final Score: {final_eval_run1.overall_score}/10")
print(f"Passed: {final_eval_run1.passed}")


print("\nFinal Draft Preview:\n", final_draft_run1[:300], "...")

=== RUN 1: TESTING END-TO-END WORKFLOW LOOP ===
[AAPL] Loaded Memory Context:
=== HISTORICAL MEMORY FOR AAPL ===
Run #1 [2026-10-01 16:28:30] - Final Quality Score: 6/10
  Summary: Final Score: 6/10 after 3 iterations. Evaluator Notes: Include specific quarterly revenue figures to enhance completeness. Ensure all data points are accurately represented and supported by the raw evidenc...
Run #2 [2026-10-01 16:28:45] - Final Quality Score: 7/10
  Summary: Final Score: 7/10 after 3 iterations. Evaluator Notes: Include specific quarterly revenue figures to enhance completeness. Ensure all stock price changes and insider transactions are accurately reflected i...
Run #3 [2026-10-01 16:30:21] - Final Quality Score: 6/10
  Summary: Final Score: 6/10 after 3 iterations. Evaluator Notes: Include specific quarterly revenue figures to enhance completeness. Ensure all sections are fully supported by evidence....
--------------------------------------------------
[AAPL] --- Starting Iteration 1/3 -

In [9]:
print("=== RUN 2: VERIFYING CROSS-RUN MEMORY INJECTION ===")

second_initial_draft = "AAPL research summary draft starting point."

final_draft_run2, final_eval_run2, history_run2 = run_evaluator_optimizer_workflow(
    ticker=TICKER,
    initial_draft=second_initial_draft,
    raw_evidence=combined_evidence_text,
    max_iterations=3,
    memory_store_path=MEMORY_FILE
)

print("\n=== RUN 2 FINAL SUMMARY ===")
print(f"Final Score: {final_eval_run2.overall_score}/10")

# Inspect the persistent memory store file to confirm JSON saved properly
test_store = ResearchMemoryStore(filepath=MEMORY_FILE)
print("\nUpdated Persistent Memory Log:")
print(test_store.get_ticker_memory(TICKER))

=== RUN 2: VERIFYING CROSS-RUN MEMORY INJECTION ===
[AAPL] Loaded Memory Context:
=== HISTORICAL MEMORY FOR AAPL ===
Run #1 [2026-10-01 16:28:45] - Final Quality Score: 7/10
  Summary: Final Score: 7/10 after 3 iterations. Evaluator Notes: Include specific quarterly revenue figures to enhance completeness. Ensure all stock price changes and insider transactions are accurately reflected i...
Run #2 [2026-10-01 16:30:21] - Final Quality Score: 6/10
  Summary: Final Score: 6/10 after 3 iterations. Evaluator Notes: Include specific quarterly revenue figures to enhance completeness. Ensure all sections are fully supported by evidence....
Run #3 [2026-10-01 16:32:57] - Final Quality Score: 7/10
  Summary: Final Score: 7/10 after 3 iterations. Evaluator Notes: Correct the stock prices for September 24 and 25, 2026, as they do not match the raw evidence. Ensure that insider transaction values are accurately r...
--------------------------------------------------
[AAPL] --- Starting Iteration 1

## 6. Test Full Workflow
Test full workflow with data from prepared datasets rather than a single company's data.

In [13]:
memory_store_path = "../data/processed/memory_store.json"

# Parse target tickers from env or default to available tickers
env_tickers = os.getenv("TARGET_TICKERS", "AAPL,NVDA")
TEST_TICKERS = [t.strip().upper() for t in env_tickers.split(",") if t.strip()]

def format_ticker_evidence(ticker: str) -> str:
    """Extracts and formats combined SEC, market, and news evidence for a ticker."""
    t_daily = daily_df[daily_df["ticker"] == ticker]
    t_news = news_df[news_df["ticker"] == ticker]
    
    evidence_block = f"""
=== DAILY MARKET & SEC FORM 4 EVIDENCE ({ticker}) ===
{t_daily.to_string(index=False) if not t_daily.empty else 'No market/SEC evidence recorded.'}

=== PROCESSED NEWS RESEARCH EVIDENCE ({ticker}) ===
{t_news.to_string(index=False) if not t_news.empty else 'No news research evidence recorded.'}
"""
    return evidence_block


# Run full workflow using data from datasets
results_summary = []

print("\n" + "="*70)
print("Starting end-to-end Evaluator-Optimizer Workflow Testing")
print("="*70)

for ticker in TEST_TICKERS:
    print(f"\nProcessing Ticker: [{ticker}]")
    
    # Format Evidence
    raw_evidence = format_ticker_evidence(ticker)
    
    # Initial Draft
    initial_draft = f"""
    Initial Research Draft for {ticker}:
    {ticker} traded over the recent period. News sentiment was generally mixed to positive.
    Revenue trends remain intact, though insider transactions and specific price changes require deeper analysis.
    """
    
    # Execute Workflow Loop
    final_report, final_eval, history = run_evaluator_optimizer_workflow(
        ticker=ticker,
        initial_draft=initial_draft,
        raw_evidence=raw_evidence,
        max_iterations=3,
        memory_store_path=str(memory_store_path)
    )
    
    # Log Test Results
    first_iter = history[0] if history else {}
    init_score = first_iter.get("overall_score") if isinstance(first_iter, dict) else getattr(first_iter, "overall_score", "N/A")
    init_missing = first_iter.get("missing_elements", []) if isinstance(first_iter, dict) else []

    results_summary.append({
        "ticker": ticker,
        "iterations_taken": len(history),
        "initial_score": init_score,
        "final_score": final_eval.overall_score,
        "passed": final_eval.passed,
        "missing_elements_addressed": len(init_missing),
        "final_report_length": len(final_report)
    })
    
    print(f"Completed [{ticker}]: Initial Score {init_score}/10 ➔ Final Score {final_eval.overall_score}/10")

# Audit summary and persistence check
print("\n" + "="*70)
print("END-TO-END TEST SUITE SUMMARY REPORT")
print("="*70)

summary_df = pd.DataFrame(results_summary)
display(summary_df)

# Verify Persistent Memory Update
memory_store = ResearchMemoryStore(filepath=str(memory_store_path))
print("\n Persistent Memory Audit (`memory_store.json`):")
for ticker in TEST_TICKERS:
    print(memory_store.get_ticker_memory(ticker))
    print("-" * 50)


Starting end-to-end Evaluator-Optimizer Workflow Testing

Processing Ticker: [AAPL]
[AAPL] Loaded Memory Context:
=== HISTORICAL MEMORY FOR AAPL ===
Run #1 [2026-10-01 16:33:25] - Final Quality Score: 9/10
  Summary: Final Score: 9/10 after 2 iterations. Evaluator Notes: The report is well-structured and clearly presents the relevant data. It accurately reflects the raw evidence, including stock prices and insider tran...
Run #2 [2026-10-01 16:35:42] - Final Quality Score: 9/10
  Summary: Final Score: 9/10 after 1 iterations. Evaluator Notes: The draft is well-structured and clearly presents the relevant information regarding AAPL's stock performance and insider transactions. Ensure that al...
Run #3 [2026-10-01 16:35:44] - Final Quality Score: 9/10
  Summary: Final Score: 9/10 after 1 iterations. Evaluator Notes: The draft is well-structured and clearly presents the relevant information. Ensure that all stock prices and insider transaction values are accurately...
-------------------

,ticker,iterations_taken,initial_score,final_score,passed,missing_elements_addressed,final_report_length
0,AAPL,2,6,9,True,3,2369
1,MSFT,2,6,9,True,2,3427



 Persistent Memory Audit (`memory_store.json`):
=== HISTORICAL MEMORY FOR AAPL ===
Run #1 [2026-10-01 16:35:42] - Final Quality Score: 9/10
  Summary: Final Score: 9/10 after 1 iterations. Evaluator Notes: The draft is well-structured and clearly presents the relevant information regarding AAPL's stock performance and insider transactions. Ensure that al...
Run #2 [2026-10-01 16:35:44] - Final Quality Score: 9/10
  Summary: Final Score: 9/10 after 1 iterations. Evaluator Notes: The draft is well-structured and clearly presents the relevant information. Ensure that all stock prices and insider transaction values are accurately...
Run #3 [2026-10-01 16:42:01] - Final Quality Score: 9/10
  Summary: Final Score: 9/10 after 2 iterations. Evaluator Notes: The report is well-structured and clearly presents the analysis of AAPL's stock performance, insider transactions, and news sentiment. Ensure that all...
--------------------------------------------------
=== HISTORICAL MEMORY FOR MSFT ===